In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon

This notebook analyses the gender bias difference for a given set of 50 runs, including statistical tests and plots.
Change the path below to analyse a specific set of runs. 

**RQ4: Experiment 1 results (Statistical tests in Section 4.3.1, Figure 4.6, Table A.8-A.11)**

This was used for all config (1, 2, 3) results, and the results from using the software engineering prompt (`\software_eng_prompt2`) used for comparison to existing methods.

In [ ]:
# Path to aggregated results
PATH = "3_Bias_Mitigation/exp1_results/comparison/swe_prompt_aggregated_results.csv"

In [ ]:
def summary_stats(x):
    x = np.asarray(x, dtype=float)
    return {
        "n": len(x),
        "mean": np.mean(x),
        "median": np.median(x),
        "min": np.min(x),
        "max": np.max(x),
        "iqr": np.percentile(x, 75) - np.percentile(x, 25),
        "std": np.std(x, ddof=1) if len(x) > 1 else 0.0,
    }


def vargha_delaney_a12(x, y):
    """
    Returns probability that a random x is greater than a random y,
    plus half probability of ties.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    more = 0
    same = 0
    for xi in x:
        more += np.sum(xi > y)
        same += np.sum(xi == y)

    return (more + 0.5 * same) / (len(x) * len(y))

def interpret_a12(a12):
    diff = abs(a12 - 0.5)
    if diff < 0.06:
        return "negligible"
    elif diff < 0.14:
        return "small"
    elif diff < 0.21:
        return "medium"
    else:
        return "large"

In [ ]:
results_df = pd.read_csv(PATH)

orig = results_df["original_gender_bias"].to_numpy()
mod = results_df["modified_gender_bias"].to_numpy()

print("\nOriginal summary:")
print(summary_stats(orig))
print("\nModified summary:")
print(summary_stats(mod))
print("\nImprovement summary:")
print(summary_stats(results_df["improvement"]))


# WILCOXON SIGNED-RANK TEST: tests whether differences consistently positive or negative
stat, p_value = wilcoxon(orig, mod, alternative="greater")
print("\nWilcoxon signed-rank test")
print(f"Statistic = {stat}")
print(f"p-value   = {p_value:.6g}")

# VARGHA-DELANEY A12
# A12 < 0.5 means modified tends to be LOWER than original
a12 = vargha_delaney_a12(mod, orig)
print("\nVargha-Delaney A12")
print(f"A12(modified, original) = {a12:.6f}")
print(f"Effect size magnitude = {interpret_a12(a12)}")

In [ ]:
# BOXPLOT
plt.figure(figsize=(6, 5))
plt.boxplot(
    [orig, mod],
    tick_labels=["Original", "Modified"]
)
plt.ylabel("Gender Bias")
plt.title("Original vs Modified Prompt Embeddings")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
# plt.savefig("gender_bias_boxplot.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# BOXPLOT (POINTS)
plt.figure(figsize=(6, 5))

data = [orig, mod]
positions = [1, 2]

plt.boxplot(data, positions=positions, widths=0.5)
rng = np.random.default_rng(42)
for i, arr in enumerate(data, start=1):
    x_jitter = rng.normal(i, 0.04, size=len(arr))
    plt.scatter(x_jitter, arr, alpha=0.6, s=25)

plt.xticks([1, 2], ["Original", "Modified"])
plt.ylabel("Gender Bias")
plt.title("Gender Bias Distribution Across Runs")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
# plt.savefig("gender_bias_boxplot_with_points.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# INTERPRETATION HELPERS
improved = np.sum(mod < orig)
same = np.sum(mod == orig)
worse = np.sum(mod > orig)
print("\nRun-by-run comparison")
print(f"Improved: {improved}/{len(mod)} ({100*improved/len(mod):.1f}%)")
print(f"Same:     {same}/{len(mod)} ({100*same/len(mod):.1f}%)")
print(f"Worse:    {worse}/{len(mod)} ({100*worse/len(mod):.1f}%)")

mean_reduction = np.mean(orig - mod)
pct_reduction = 100 * mean_reduction / np.mean(orig)
print(f"\nMean absolute reduction in gender bias = {mean_reduction:.4f}")
print(f"Mean percentage reduction in gender bias = {pct_reduction:.2f}%")